In [1]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:

songs_data_path = "../data/Music_Info.csv"
users_data_path =  "../data/User_Listening_History.csv"

In [3]:
songs_df = pd.read_csv(songs_data_path,usecols =['track_id','name','artist','spotify_preview_url'])
songs_df.head()

,track_id,name,artist,spotify_preview_url
0,TRIOREW128F424EAF0,Mr. Brightside,The Killers,https://p.scdn.co/mp3-preview/4d26180e6961fd46...
1,TRRIVDJ128F429B0E8,Wonderwall,Oasis,https://p.scdn.co/mp3-preview/d012e536916c927b...
2,TROUVHL128F426C441,Come as You Are,Nirvana,https://p.scdn.co/mp3-preview/a1c11bb1cb231031...
3,TRUEIND128F93038C4,Take Me Out,Franz Ferdinand,https://p.scdn.co/mp3-preview/399c401370438be4...
4,TRLNZBD128F935E4D8,Creep,Radiohead,https://p.scdn.co/mp3-preview/e7eb60e9466bc3a2...


In [4]:
! pip install dask[dataframe]

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import dask.dataframe as dd

df = dd.read_csv(users_data_path)
df.head(5)

,track_id,user_id,playcount
0,TRIRLYL128F42539D1,b80344d063b5ccb3212f76538f3d9e43d87dca9e,1
1,TRFUPBA128F934F7E1,b80344d063b5ccb3212f76538f3d9e43d87dca9e,1
2,TRLQPQJ128F42AA94F,b80344d063b5ccb3212f76538f3d9e43d87dca9e,1
3,TRTUCUY128F92E1D24,b80344d063b5ccb3212f76538f3d9e43d87dca9e,1
4,TRHDDQG12903CB53EE,b80344d063b5ccb3212f76538f3d9e43d87dca9e,1


In [6]:
df

,track_id,user_id,playcount
npartitions=9,,,
,string,string,int64
,...,...,...
...,...,...,...
,...,...,...
,...,...,...


In [7]:
df

,track_id,user_id,playcount
npartitions=9,,,
,string,string,int64
,...,...,...
...,...,...,...
,...,...,...
,...,...,...


In [8]:
! pip install graphviz

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
df.npartitions

9

In [10]:
unique_track = df.loc[:,'track_id'].nunique()

In [11]:
unique_track = unique_track.compute()
unique_track

np.int64(30459)

In [12]:
unique_users = df.loc[:,'user_id'].nunique()
unique_users = unique_users.compute()
unique_users

np.int64(962037)

In [13]:
unique_track_ids = df.loc[:,'track_id'].unique().compute()
unique_track_ids = unique_track_ids.tolist()
unique_track_ids 

['TRLXSNR128F429361D',
 'TRPUGUW128F426BF6F',
 'TRISTWT128F1488FBB',
 'TRKPWGR128E078EE06',
 'TRXQJWK128F146DF76',
 'TRGASNY128F14696B0',
 'TRSQWMI128F932FC8A',
 'TRBVNWT128F93173BA',
 'TRBHQZM128F42A52D2',
 'TRSWJHK128F429EA6F',
 'TRRUZLF128F42396D0',
 'TRMPCGW128F92E0670',
 'TRJNMNC128F427ED16',
 'TRWCIAX128F42925BD',
 'TRGVKBK128F429EA2D',
 'TRGRPEU128F932BD78',
 'TRANEZZ128F92FFC06',
 'TRRYLYK128F92F89F6',
 'TRQGHWL128EF33FB48',
 'TRWZFIC128F933BCA3',
 'TRKSEEY12903CCD312',
 'TRBTCYJ128F92F9586',
 'TRQPRPK12903CCF3B6',
 'TRJSQQT128F149F9B4',
 'TREWPIM128F4261B51',
 'TRMZPXZ128F92F3495',
 'TRTGEVW128F14979BB',
 'TROUMHD128F9355B89',
 'TRKRNZF12903CB52BC',
 'TRBSBCX128F92DEF11',
 'TRXHDTA128F42A077A',
 'TRSJBLT128F429EA02',
 'TRTJYDU128F92E49EE',
 'TRDTMGK12903CC557F',
 'TRBOAJY128F14979B5',
 'TRBYDXV128F424888B',
 'TRCIOVJ128F429EB51',
 'TRYBFNR128F426BE3D',
 'TRWOYHS128F931EB5A',
 'TRNNLYM128F92EDE7F',
 'TRATHTH128F42BC347',
 'TRXERRS128F42969E4',
 'TRZPVLJ128F148D2F7',
 'TRGTVVB12

In [14]:
filtered_songs = songs_df[songs_df['track_id'].isin(unique_track_ids)]
filtered_songs.reset_index(drop = True , inplace = True)

In [15]:
filtered_songs.shape

(30459, 4)

In [27]:

import dask.dataframe as dd
import numpy as np
from scipy.sparse import csr_matrix

# Step 1: Load data with Dask
# Assume the dataset is in a CSV file

df = dd.read_csv(users_data_path)

# Step 2: Ensure playcount is numeric
df['playcount'] = df['playcount'].astype(np.float64)
df = df.categorize(columns=['user_id', 'track_id'])

# Step 3: Convert user_id and track_id to numeric indices
# This is necessary for creating a sparse matrix later
user_mapping = df['user_id'].cat.codes
track_mapping = df['track_id'].cat.codes

df = df.assign(
    user_idx=user_mapping,
    track_idx=track_mapping
)
     

In [28]:
df.head()

,track_id,user_id,playcount,user_idx,track_idx
0,TRIRLYL128F42539D1,b80344d063b5ccb3212f76538f3d9e43d87dca9e,1.0,691377,10705
1,TRFUPBA128F934F7E1,b80344d063b5ccb3212f76538f3d9e43d87dca9e,1.0,691377,7334
2,TRLQPQJ128F42AA94F,b80344d063b5ccb3212f76538f3d9e43d87dca9e,1.0,691377,14212
3,TRTUCUY128F92E1D24,b80344d063b5ccb3212f76538f3d9e43d87dca9e,1.0,691377,23206
4,TRHDDQG12903CB53EE,b80344d063b5ccb3212f76538f3d9e43d87dca9e,1.0,691377,8936


In [29]:
interaction_array = df.groupby(['track_idx','user_idx'])['playcount'].sum().reset_index()

In [30]:
interaction_array.head(10)

,track_idx,user_idx,playcount
0,0,15780,3.0
1,0,76968,1.0
2,0,134525,2.0
3,0,231541,1.0
4,0,305348,1.0
5,0,587045,1.0
6,0,591458,1.0
7,0,593674,1.0
8,0,610079,2.0
9,0,611454,1.0


In [31]:
row_indices = interaction_array['track_idx']
col_indices = interaction_array['user_idx']
values = interaction_array['playcount']

In [32]:
n_tracks = unique_track
n_users = unique_users

sparse_matrix = csr_matrix((values , (row_indices , col_indices)) , shape = (n_tracks , n_users))

In [33]:
from sklearn.metrics.pairwise import cosine_similarity

In [34]:
np.where(df['track_id'].cat.categories == 'TRIRLYL128F42539D1')

(array([10705]),)

In [53]:
def collaborative_recommendation(song_name,user_data,songs_data,interaction_matrix,k=10):
    # fetch the row from songs data
    song_row = songs_data[songs_data["name"] == song_name]
    print(song_row)
    # track_id of input song
    input_track_id = song_row['track_id'].values.item()
    print(input_track_id)
    # index value of track_id
    ind = np.where(user_data['track_id'].cat.categories == input_track_id)[0].item()
    print(ind)
    # fetch the input vector
    input_array = interaction_matrix[ind]
    # get similarity scores
    similarity_scores = cosine_similarity(input_array, interaction_matrix)
    # get top k recommendations
    recommendation_track_ids = df['track_id'].cat.categories[np.argsort(similarity_scores.ravel())[-k-1:][::-1]]
    print(recommendation_track_ids)
    # get top scores
    top_scores = np.sort(similarity_scores.ravel())[-k-1:][::-1]
    print(top_scores)
    # get the songs from data and print
    temp_df = pd.DataFrame({"track_id":recommendation_track_ids.tolist(),
                            "score":top_scores})
    print(temp_df)
    top_k_songs = (
                    songs_data
                    .loc[songs_data["track_id"].isin(recommendation_track_ids)]
                    .merge(temp_df,on="track_id")
                    .sort_values(by="score",ascending=False)
                    .drop(columns=["track_id","score"])
                    .reset_index(drop=True)
                    )
    return top_k_songs

In [54]:

collaborative_recommendation(song_name="Crazy in Love",
                             user_data=df,
                             songs_data=filtered_songs,
                             interaction_matrix=sparse_matrix)

                track_id           name   artist  \
3337  TROINZB128F932F740  Crazy in Love  Beyoncé   

                                    spotify_preview_url  
3337  https://p.scdn.co/mp3-preview/807828ea7070bda7...  
TROINZB128F932F740
17018
Index(['TRZZZRJ128F42819AF', 'TRZZZHL128F9329CFB', 'TRZZZCN128F9317A03',
       'TRZZZCL128F428BB80', 'TRZZYMU128E0792400', 'TRZZXVN128F93285B4',
       'TRZZXOQ128F932A083', 'TRZZXJT128F931D72C', 'TRZZXIR128F9308AD4',
       'TRZZVMG128F149B9A2', 'TRZZUTD12903CADD68'],
      dtype='string', name='track_id')
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
              track_id  score
0   TRZZZRJ128F42819AF    0.0
1   TRZZZHL128F9329CFB    0.0
2   TRZZZCN128F9317A03    0.0
3   TRZZZCL128F428BB80    0.0
4   TRZZYMU128E0792400    0.0
5   TRZZXVN128F93285B4    0.0
6   TRZZXOQ128F932A083    0.0
7   TRZZXJT128F931D72C    0.0
8   TRZZXIR128F9308AD4    0.0
9   TRZZVMG128F149B9A2    0.0
10  TRZZUTD12903CADD68    0.0


,name,artist,spotify_preview_url
0,Solo Dolo (Nightmare),Kid Cudi,https://p.scdn.co/mp3-preview/66373133ca4e7ebc...
1,After,Ihsahn,https://p.scdn.co/mp3-preview/d566307f9eb10478...
2,Day Five: Voices,Ayreon,https://p.scdn.co/mp3-preview/9f62f3a9d9a232a6...
3,Big Wiggly Style,The Devil Wears Prada,https://p.scdn.co/mp3-preview/39bf3d8d7d027e44...
4,Dead Season,Nujabes,https://p.scdn.co/mp3-preview/14d665ba14f7b548...
5,Lord Anthony,Belle and Sebastian,https://p.scdn.co/mp3-preview/c7835d4b82ca1392...
6,Cover Up,Imagine Dragons,https://p.scdn.co/mp3-preview/382d85deaca4e3e4...
7,The Ship of Pills and Needed Things,I Am Ghost,https://p.scdn.co/mp3-preview/331a9b39765bade1...
8,Flutter Girl,Chris Cornell,https://p.scdn.co/mp3-preview/086e8821cd2cf926...
9,Abschied,Sopor Aeternus & The Ensemble of Shadows,https://p.scdn.co/mp3-preview/3d825e9605292f57...
